# arXiv 논문 검색 클라이언트 (Google Colab용)

**용도**: 팀 공유 서버(EC2 FastAPI)에 접속해 논문 검색/조회를 하는 팀원용 노트북. 저장소 클론 없이 **이 노트북 하나만으로** 코랩에서 바로 실행됩니다.

**전체 프로세스에서의 역할**: `client_example.py`의 코랩 버전 — 관심사 프로필로 유사 논문을 검색해 벤치마크 라벨링 후보를 추출하는 데 사용합니다.

**실행 방법**:
1. 이 파일을 [Google Colab](https://colab.research.google.com)에서 열기 (GitHub에서 열기 또는 업로드)
2. 아래 셀을 순서대로 실행
3. 접속 정보 셀에서 팀에서 공유받은 `API_BASE`와 `API_KEY` 입력

**사전 조건**: EC2 서버(`uvicorn app:app`)가 켜져 있어야 하고, 팀에서 공유받은 서버 주소와 API 키가 필요합니다.

> ⚠️ **API 키를 노트북에 직접 타이핑하지 마세요.** 아래 셀이 실행 시 입력창(`getpass`)으로 받기 때문에 키가 노트북 파일에 저장되지 않습니다. 이 노트북은 GitHub에 커밋되는 파일입니다.

## 1. 접속 정보 설정

- `API_BASE`: 서버 주소 (예: `http://<EC2-IP>:8000`) — 비밀값은 아니지만 팀 외부 공유 금지
- `API_KEY`: 실행하면 나오는 입력창에 붙여넣기 (화면에 표시되지 않음)

코랩 **보안 비밀**(왼쪽 🔑 아이콘)에 `ARXIV_API_BASE`, `ARXIV_API_KEY`를 등록해두면 입력 없이 자동으로 불러옵니다.

In [ ]:
from getpass import getpass

API_BASE = None
API_KEY = None

# 1순위: 코랩 보안 비밀(Secrets)에서 읽기
try:
    from google.colab import userdata
    try:
        API_BASE = userdata.get("ARXIV_API_BASE")
        API_KEY = userdata.get("ARXIV_API_KEY")
        print("코랩 보안 비밀에서 접속 정보를 불러왔습니다.")
    except Exception:
        pass
except ImportError:
    pass  # 코랩이 아닌 환경 (로컬 주피터 등)

# 2순위: 직접 입력
if not API_BASE:
    API_BASE = input("API_BASE (예: http://1.2.3.4:8000): ").strip()
if not API_KEY:
    API_KEY = getpass("API_KEY (입력해도 화면에 안 보임): ").strip()

API_BASE = API_BASE.rstrip("/")
HEADERS = {"X-API-Key": API_KEY}
print(f"서버: {API_BASE}")

## 2. 클라이언트 함수 정의

`client_example.py`와 동일한 함수들입니다 (requests만 사용, 무거운 라이브러리 불필요).

In [ ]:
import requests
import pandas as pd


def health():
    """서버 생존 확인 + 전체 논문 수 (인증 불필요)."""
    resp = requests.get(f"{API_BASE}/health", timeout=10)
    resp.raise_for_status()
    return resp.json()


def search(query: str, top_k: int = 20, category: str = None):
    """관심사 프로필 텍스트로 유사 논문 검색.

    Args:
        query: 검색 텍스트 (영어 권장 — 임베딩 모델이 영어 모델)
        top_k: 반환 논문 수
        category: 주 카테고리 필터 (예: "cs.RO"), 생략 시 전체

    Returns:
        논문 dict 리스트. score는 cosine distance — **작을수록 유사**.
    """
    payload = {"query": query, "top_k": top_k}
    if category:
        payload["category"] = category
    resp = requests.post(f"{API_BASE}/search", json=payload, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_paper(arxiv_id: str):
    """단일 논문 상세 조회 (초록 원문 포함)."""
    resp = requests.get(f"{API_BASE}/papers/{arxiv_id}", headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_papers(arxiv_ids: list):
    """여러 논문 상세 조회 (라벨링 후보 리스트 확인용)."""
    resp = requests.get(
        f"{API_BASE}/papers",
        params={"ids": ",".join(arxiv_ids)},
        headers=HEADERS,
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def search_df(query: str, top_k: int = 20, category: str = None) -> pd.DataFrame:
    """search() 결과를 보기 좋은 DataFrame으로 반환."""
    results = search(query, top_k=top_k, category=category)
    df = pd.DataFrame(results)
    if df.empty:
        return df
    cols = ["arxiv_id", "score", "title", "primary_category", "submitted_date", "abs_url"]
    return df[[c for c in cols if c in df.columns]]


print("함수 정의 완료")

## 3. 접속 확인

In [ ]:
health()
# {'status': 'ok', 'paper_count': 60000+} 형태가 나오면 정상

## 4. 논문 검색

- **쿼리는 영어로** 작성하세요 (임베딩 모델 `bge-small-en`이 영어 모델)
- `score`는 cosine **거리**라서 **작을수록 유사**합니다 (정렬 방향 주의)

In [ ]:
# 간단한 주제 검색
search_df("speculative decoding for efficient LLM inference", top_k=10, category="cs.CL")

## 5. 관심사 프로필로 라벨링 후보 추출

노션 "관심사 프로필 10개 제안" 페이지의 프로필을 그대로 넣어 후보 풀을 뽑는 예시입니다.
프로필 본문(영어)을 쿼리로 사용하고, 라벨링용으로는 top 30~50 정도를 넉넉히 추출합니다.

In [ ]:
# 예: P1. 언어 조건 로봇 매니퓰레이션 프로필
profile_p1 = (
    "I am interested in language-conditioned robot manipulation, "
    "especially how vision-language-action models interpret ambiguous or "
    "underspecified natural language instructions and ground them into "
    "executable actions."
)

candidates = search_df(profile_p1, top_k=30, category="cs.RO")
candidates

In [ ]:
# 후보 목록을 CSV로 저장 (라벨링 시트 만들 때 사용)
# 코랩 왼쪽 파일 탭에서 다운로드하거나 구글 드라이브에 복사하면 됩니다.
candidates.to_csv("labeling_candidates_p1.csv", index=False)
print("labeling_candidates_p1.csv 저장 완료 —", len(candidates), "건")

## 6. 논문 상세 조회

검색 결과에서 궁금한 논문의 초록 전문을 확인할 때 사용합니다.

In [ ]:
# 위 검색 결과 중 첫 번째 논문의 상세 정보
if len(candidates) > 0:
    detail = get_paper(candidates.iloc[0]["arxiv_id"])
    print("제목:", detail["title"])
    print("카테고리:", detail["categories"])
    print("제출일:", detail["submitted_date"])
    print("링크:", detail["abs_url"])
    print()
    print(detail["abstract_clean"][:800])

## 자주 겪는 문제

| 증상 | 원인 | 해결 |
|---|---|---|
| `ConnectionError` / 타임아웃 | 서버가 꺼져 있거나 주소 오타 | EC2에서 uvicorn 구동 상태, `API_BASE` 확인 |
| `401 Unauthorized` | API 키 불일치 | 팀 공유 키 재확인 후 1번 셀 다시 실행 |
| `422 Unprocessable Entity` | 헤더 누락 등 요청 형식 문제 | 1~2번 셀을 순서대로 재실행 |
| 검색 결과가 이상함 | 한국어 쿼리 | 영어로 쿼리 작성 |
| 상위 결과가 안 비슷함 | score를 유사도로 착각 | score는 거리 — 작을수록 유사 |